# Stage 1b: Build full data workbooks for subsequent analysis

Place this notebook in the main kinetics directory and run it there. It recursively finds `*-setup_reformat.xlsx` files, validates the existing `nu_id` and `nu_name` columns, joins the kinetic `k_obs` and `k_obs_method` columns from each matching `exp_initial_rate_outputs/*_exp_initial_rate_fit_results.xlsx` workbook, and writes `NSXXX_fulldata.xlsx` next to the reformatted setup workbook.

In [ ]:
from pathlib import Path
import re
import traceback

import pandas as pd
from openpyxl import load_workbook

pd.set_option("display.max_columns", 100)

SEARCH_ROOT = Path.cwd()
KINETIC_SHEET = "Comparison_Final_kobs"
KINETIC_COLUMNS = ["k_obs", "k_obs_method"]

CONCENTRATION_COLUMNS = ["[nu]tot", "[nu_prot]", "[nacl]_manual", "[nacl]_ionize", "[nacl]_final", "[ha]tot", "[PX]", "[edta]"]
IONIC_STRENGTH_COLUMNS = ["i_rxn_man", "i_rxn_ionize", "i_rxn_final"]
PH_OR_PKA_COLUMNS = ["ha_pka", "ha_initial_pH", "ha_pka_corr", "nu_pka_corr", "pH_corr_manual", "pH_corr_ionize", "pH_final"]
CHARGE_COLUMNS = ["ha_chrg", "a_chrg", "nu_chrg"]

setup_reformat_files = sorted(
    p for p in SEARCH_ROOT.glob("**/*-setup_reformat.xlsx")
    if not p.name.startswith("~$") and "ipynb_checkpoints" not in p.parts
)

print(f"Search root: {SEARCH_ROOT}")
print(f"Found {len(setup_reformat_files)} reformatted setup workbook(s).")
for path in setup_reformat_files:
    print("-", path.relative_to(SEARCH_ROOT))

## Helpers

In [ ]:
def validate_nucleophile_columns(df):
    missing = [column for column in ("nu_id", "nu_name") if column not in df.columns]
    if missing:
        raise ValueError(f"Setup reformat workbook is missing nucleophile columns: {missing}")
    out = df.copy()
    out["nu_id"] = out["nu_id"].astype("string").str.strip()
    out["nu_name"] = out["nu_name"].astype("string").str.strip()
    blank = out["nu_id"].isna() | out["nu_name"].isna() | out["nu_name"].eq("")
    if blank.any():
        raise ValueError(f"Setup reformat workbook contains {int(blank.sum())} blank nucleophile identity row(s)")
    invalid_ids = ~out["nu_id"].str.fullmatch(r"Nu\d+")
    if invalid_ids.any():
        values = sorted(out.loc[invalid_ids, "nu_id"].unique().tolist())
        raise ValueError(f"Invalid nucleophile identifiers: {values}")
    id_name_counts = out.groupby("nu_id", dropna=False)["nu_name"].nunique()
    name_id_counts = out.groupby("nu_name", dropna=False)["nu_id"].nunique()
    if (id_name_counts > 1).any() or (name_id_counts > 1).any():
        raise ValueError("Nucleophile identifiers and names are not one-to-one")
    return out


def infer_experiment_id(path, setup_df=None):
    if setup_df is not None and "expnr" in setup_df.columns:
        values = setup_df["expnr"].dropna().astype(str).str.strip()
        if not values.empty:
            match = re.search(r"NS\d+", values.iloc[0], flags=re.IGNORECASE)
            return match.group(0).upper() if match else values.iloc[0]
    match = re.search(r"(NS\d+)", path.name, flags=re.IGNORECASE)
    return match.group(1).upper() if match else path.stem.replace("-setup_reformat", "")


def find_kinetic_file(setup_path, expnr):
    output_dir = setup_path.parent / "exp_initial_rate_outputs"
    if not output_dir.exists():
        raise FileNotFoundError(f"Missing exp_initial_rate_outputs folder next to {setup_path.name}")
    candidates = sorted(output_dir.glob(f"*{expnr}*exp_initial_rate_fit_results*.xlsx"))
    candidates = [p for p in candidates if not p.name.startswith("~$")]
    if not candidates:
        candidates = [p for p in sorted(output_dir.glob("*exp_initial_rate_fit_results*.xlsx")) if not p.name.startswith("~$")]
    if not candidates:
        raise FileNotFoundError(f"No kinetic fit results workbook found in {output_dir}")
    if len(candidates) > 1:
        exact = [p for p in candidates if expnr in p.name]
        if len(exact) == 1:
            return exact[0]
        raise ValueError("Multiple kinetic fit result workbooks found: " + ", ".join(p.name for p in candidates))
    return candidates[0]


def read_kinetics(path):
    df = pd.read_excel(path, sheet_name=KINETIC_SHEET, engine="openpyxl")
    missing = [col for col in ["Entry"] + KINETIC_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing column(s) in {KINETIC_SHEET}: {missing}")
    keep = df[["Entry"] + KINETIC_COLUMNS].copy()
    keep = keep.loc[keep["Entry"].notna()].copy()
    keep["Entry"] = keep["Entry"].astype(str).str.strip()
    if keep["Entry"].duplicated().any():
        duplicates = sorted(keep.loc[keep["Entry"].duplicated(), "Entry"].unique().tolist())
        raise ValueError(f"Duplicate kinetic Entry values: {duplicates[:10]}")
    return keep


def apply_number_formats(path):
    wb = load_workbook(path)
    ws = wb.active
    header_to_col = {cell.value: cell.column for cell in ws[1]}
    for name in CONCENTRATION_COLUMNS + IONIC_STRENGTH_COLUMNS + ["k_obs"]:
        if name in header_to_col:
            col = header_to_col[name]
            for row in range(2, ws.max_row + 1):
                ws.cell(row, col).number_format = "0.00E+00"
    for name in PH_OR_PKA_COLUMNS:
        if name in header_to_col:
            col = header_to_col[name]
            for row in range(2, ws.max_row + 1):
                ws.cell(row, col).number_format = "0.00"
    for name in CHARGE_COLUMNS:
        if name in header_to_col:
            col = header_to_col[name]
            for row in range(2, ws.max_row + 1):
                ws.cell(row, col).number_format = "0"
    for column_cells in ws.columns:
        header = column_cells[0].value
        if header in {"nu_id", "nu_name", "ha_type", "k_obs_method"}:
            ws.column_dimensions[column_cells[0].column_letter].width = 24
        elif header in {"electrophile_type"}:
            ws.column_dimensions[column_cells[0].column_letter].width = 16
        else:
            ws.column_dimensions[column_cells[0].column_letter].width = 14
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    wb.save(path)

## Build full data workbooks

In [ ]:
def build_fulldata(setup_reformat_path):
    setup_df = pd.read_excel(setup_reformat_path, engine="openpyxl")
    if "entry" not in setup_df.columns:
        raise ValueError("Setup reformat workbook has no 'entry' column")
    setup_df = validate_nucleophile_columns(setup_df)

    expnr = infer_experiment_id(setup_reformat_path, setup_df)
    kinetic_path = find_kinetic_file(setup_reformat_path, expnr)
    kinetic = read_kinetics(kinetic_path)

    full = setup_df.copy()
    full["entry"] = full["entry"].astype(str).str.strip()
    full = full.merge(kinetic, how="left", left_on="entry", right_on="Entry")
    full = full.drop(columns=["Entry"])

    unmatched = int(full["k_obs"].isna().sum())
    output_path = setup_reformat_path.with_name(f"{expnr}_fulldata.xlsx")
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        full.to_excel(writer, sheet_name="Tabelle1", index=False)
    apply_number_formats(output_path)

    return output_path, kinetic_path, full, unmatched


results = []
for setup_path in setup_reformat_files:
    try:
        output_path, kinetic_path, full, unmatched = build_fulldata(setup_path)
        warnings = []
        if unmatched:
            warnings.append(f"{unmatched} row(s) have no matched k_obs")
        results.append({
            "source": str(setup_path.relative_to(SEARCH_ROOT)),
            "kinetic_source": str(kinetic_path.relative_to(SEARCH_ROOT)),
            "output": str(output_path.relative_to(SEARCH_ROOT)),
            "status": "ok" if not warnings else "ok_with_warnings",
            "rows": len(full),
            "columns": len(full.columns),
            "warnings": "; ".join(warnings),
            "error": "",
        })
        print(f"OK: {setup_path.relative_to(SEARCH_ROOT)} -> {output_path.name}")
        if warnings:
            print("  warnings:", "; ".join(warnings))
    except Exception as exc:
        results.append({
            "source": str(setup_path.relative_to(SEARCH_ROOT)),
            "kinetic_source": "",
            "output": "",
            "status": "error",
            "rows": 0,
            "columns": 0,
            "warnings": "",
            "error": f"{type(exc).__name__}: {exc}",
        })
        print(f"ERROR: {setup_path.relative_to(SEARCH_ROOT)}")
        print(traceback.format_exc())

report = pd.DataFrame(results)
display(report)